# 03 - FAIR & Semantic Data Layer

**Goal:** make our dataset **F**indable, **A**ccessible, **I**nteroperable, **R**eusable, then query it semantically.

**What we will do:**
1. Inspect machine-readable **JSON-LD metadata** (Schema.org + DCAT)
2. Build an **RDF knowledge graph** (materials structures properties provenance)
3. Query it with **SPARQL** (the 'SQL' of knowledge graphs)

In [1]:
import json
from pathlib import Path

from rdflib import Graph
from rdflib.namespace import RDF

ROOT = Path('..').resolve()

## 1. FAIR metadata (JSON-LD)

JSON-LD wraps data in a JSON `@context` that maps every term to a standard vocabulary URI (e.g. `schema:license` means `https://schema.org/license`). A machine can dereference these and learn the meaning without asking a human.

In [2]:
with open(ROOT / 'metadata' / 'dataset_metadata.jsonld') as f:
    metadata = json.load(f)

print('Vocabularies used:', list(metadata['@context'].keys()))
print()
print('Declared as:', metadata['@type'])
print('License:', metadata['schema:license']['schema:name'])
print('Variables measured (interoperable units):')
for v in metadata['schema:variableMeasured']:
    print(f"  - {v['schema:name']} ({v.get('schema:unitText', 'n/a')})")

Vocabularies used: ['schema', 'dcat', 'dcterms', 'prov', 'xsd', 'dpv']

Declared as: ['dcat:Dataset', 'schema:Dataset']
License: CC-BY-4.0
Variables measured (interoperable units):
  - band_gap (eV)
  - formation_energy_per_atom (eV/atom)
  - density (g/cm3)
  - crystal_system (n/a)


## 2. Build the knowledge graph

Every material becomes an entity `matl:mp-XXXX` typed as `mat:Material`, connected by **typed predicates** (URIs from `src/vocab.py`). Provenance uses the W3C **PROV-O** standard.

In [3]:
import sys
sys.path.insert(0, str(ROOT))
from src.rdf import build_graph
from src.data import load_raw_data

df = load_raw_data(ROOT / 'data' / 'processed' / 'mp_oxides_clean.csv')
g = build_graph(df)
print(f'Graph contains {len(g)} triples for {len(df)} materials')
print()

# Show triples for one material
mat_uri = 'https://fair-data-learning.example.org/entity/material/' + df.iloc[0]['material_id']
print(f'Triples for {df.iloc[0]["material_id"]}:')
for s, p, o in g.triples((None, None, None)):
    if str(s) == mat_uri:
        print(f'  {p.split("/")[-1]:<28} {o}')

Graph contains 12196 triples for 1000 materials

Triples for mp-1523737:
  materials#hasFormationEnergyPerAtom 0.0901382308333325
  22-rdf-syntax-ns#type        https://fair-data-learning.example.org/ontology/materials#Material
  materials#containsElement    http://www.wikidata.org/entity/Q627
  materials#hasSpaceGroup      https://fair-data-learning.example.org/entity/space_group/P2_1/c
  materials#containsElement    http://www.wikidata.org/entity/Q688
  materials#hasDensity         1.8124752360018217
  prov#wasDerivedFrom          https://materialsproject.org/materials/mp-1523737
  materials#hasCrystalSystem   https://fair-data-learning.example.org/entity/crystal_system/Monoclinic
  materials#isStable           false
  rdf-schema#label             NClO
  materials#hasFormula         NClO
  materials#containsElement    http://www.wikidata.org/entity/Q629
  materials#hasBandGap         2.0666


In [4]:
ttl = ROOT / 'data' / 'processed' / 'knowledge_graph.ttl'
g.serialize(destination=ttl, format='turtle')
print(f'Knowledge graph saved: {ttl} (interoperable Turtle/RDF format)')

Knowledge graph saved: /home/hossein/fair-data-learning/data/processed/knowledge_graph.ttl (interoperable Turtle/RDF format)


## 3. Query with SPARQL

SPARQL matches **graph patterns**. `?m a mat:Material` binds `?m` to every material node. Combined with a band-gap filter, one query answers a materials-science question.

### Query A: wide-band-gap (> 2.5 eV) compounds

In [5]:
q_wide = """
SELECT ?formula (MAX(?gap) AS ?max_gap)
WHERE {
  ?m a mat:Material ;
     mat:hasFormula ?formula ;
     mat:hasBandGap ?gap .
  FILTER(?gap >= 2.5)
}
GROUP BY ?formula
ORDER BY DESC(?max_gap)
LIMIT 8
"""

for row in g.query(q_wide, initNs={'mat': 'https://fair-data-learning.example.org/ontology/materials#'}):
    print(f'  {row["formula"]:<12} max gap = {float(row["max_gap"]):.2f} eV')

  BeO          max gap = 8.17 eV
  N2O          max gap = 6.24 eV
  CO           max gap = 6.15 eV
  H2O          max gap = 6.13 eV
  H2CO         max gap = 6.03 eV
  H3ClO        max gap = 5.50 eV
  YbOF         max gap = 5.45 eV
  H4C2O        max gap = 5.41 eV


In [6]:
# Query B: materials containing iron, joined via Wikidata URIs (wd:Q677)
q_iron = """
SELECT DISTINCT ?formula ?gap
WHERE {
  ?m a mat:Material ;
     mat:containsElement wd:Q677 ;
     mat:hasFormula ?formula ;
     mat:hasBandGap ?gap .
}
ORDER BY DESC(?gap)
LIMIT 6
"""

for row in g.query(q_iron, initNs={'mat': 'https://fair-data-learning.example.org/ontology/materials#', 'wd': 'http://www.wikidata.org/entity/'}):
    print(f'  {row["formula"]:<12} band gap = {float(row["gap"]):.2f} eV')

  Fe4OF6       band gap = 2.79 eV
  Fe4OF6       band gap = 2.77 eV
  Fe4OF6       band gap = 2.63 eV
  Fe3OF7       band gap = 2.57 eV
  Fe3OF7       band gap = 2.53 eV
  Fe4OF6       band gap = 2.44 eV


In [7]:
# Query C: provenance — which Materials Project page did each material come from?
q_prov = """
SELECT ?formula ?source
WHERE {
  ?m a mat:Material ;
     mat:hasFormula ?formula ;
     prov:wasDerivedFrom ?source .
}
LIMIT 5
"""
for row in g.query(q_prov, initNs={'mat': 'https://fair-data-learning.example.org/ontology/materials#', 'prov': 'https://www.w3.org/ns/prov#'}):
    print(f'  {row["formula"]:<10} <- {row["source"]}')

  NClO       <- https://materialsproject.org/materials/mp-1523737
  V3OF7      <- https://materialsproject.org/materials/mp-1567687
  V3OF7      <- https://materialsproject.org/materials/mp-1568124
  Fe4OF6     <- https://materialsproject.org/materials/mp-1568829
  Fe4OF6     <- https://materialsproject.org/materials/mp-1569012


## Summary

- **Findable:** JSON-LD metadata with identifiers + keywords
- **Accessible:** public repo + open formats (CSV, Turtle/RDF)
- **Interoperable:** RDF triples using Schema.org/DCAT/PROV vocabularies
- **Reusable:** license, provenance chain, typed properties

Next steps toward 'semantic MLOps': link the trained model to the dataset in the graph, publish datasets with a DOI (Zenodo), and serve the graph via a SPARQL endpoint.